# Exercise 1. Prepare Big (Text) Data for ML!
In the sections below, we will explore the dataset and perform the pre-processing needed for machine learning - regardless of whether it is text or not. After completing this exercise, we'll be ready to apply `Bag-of-Words` or `TF-IDF` vectorization to our `text`!

## 1.1 Introducing the Data

As mentioned, you will be working with {cite:t}`dugan-etal-2024-raid`'s RAID dataset. The files can be found in `resources/data/raid` on `UCloud` (should be mounted if you followed [Class Setup](class-setup)):
```bash
└── raid
    ├── test.csv
    ├── test_none.csv <-- NEVER EVALUATE ON TEST BEFORE BEING DONE WITH TRAIN!!
    ├── train.csv
    └── train_none.csv
```

You will be working with the `train_none.csv` which contains the following:
```{figure} ../figures/class2/raid_figure.png
---
name: raid-overview
---
Figure modified from {cite:t}`dugan-etal-2024-raid`.
```

`train_none.csv` is a subset of the entire dataset. The full dataset also contains `adversarial attacks` (6.2M examples in total!). For simplicity, we won't be looking at those today.

:::{admonition} What are adversarial attacks?
:class: dropdown, tip
Adversarial attacks are carefully designed modifications to input data (in our case, text) that can cause a classifier to make incorrect predictions. They exploit weaknesses in the model that were not encountered during training. Incorporating such attacks into the training process can improve the classifier’s robustness against unexpected or manipulated inputs in real-world settings.

In {cite:t}`dugan-etal-2024-raid`'s RAID, these include everything from British spelling, article deletions, and mispellings. Read more about it in their paper!

I have downloaded the full dataset `train.csv` for you to explore if you like. It contains 6.2M examples, so consider choosing a larger UCloud machine if it loads slowly.
:::

### Load the Data
Start by importing `pathlib` and `pandas`:

In [6]:
from pathlib import Path
import pandas as pd

Define paths:

In [7]:
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

In [8]:
raw_df = pd.read_csv(data_path)

Let's look at how many rows there is in our df:

In [9]:
print(len(raw_df))

467985


### Your Turn: Look at the Raw Data
```{admonition} HANDS-ON
:class: red

1. Load `raw_df` in your notebook if you haven't already! 
2. Print all column names  `raw_df` 
3. Do you notice any columns that you might not immediately know what corresponds to? Read up on [the column names](https://huggingface.co/datasets/liamdugan/raid#data-fields) before proceeding!
4. From {numref}`raid-overview`, we have gotten an overview of the kinds of LLMs used in this dataset, but what are they called in our dataframe? Find all unique values in the `models` column.
```

#### Print Column Names

```{admonition} HINT
:class: tip, dropdown
Look at the .columns attribute on Pandas - see [docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) for help
```

Check the solution:

In [10]:
columns_in_df = raw_df.columns.tolist() # you don't technically need .tolist() - it it just to get it in a neat list (try to remove to see effect) 
print(columns_in_df)

['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']


#### Unique Models

```{admonition} HINT
:class: tip, dropdown
What is the LLM column called? You need this, and then you can use the `.unique()` method:
https://pandas.pydata.org/docs/reference/api/pandas.unique.html
```

Solution below:

In [11]:
unique_models = raw_df["model"].unique()
print(unique_models)

['human' 'llama-chat' 'mpt' 'mpt-chat' 'gpt2' 'mistral' 'mistral-chat'
 'gpt3' 'cohere' 'chatgpt' 'gpt4' 'cohere-chat']


## 1.2 Subset Data
We want to classify `human` versus `chatgpt` which we also call our *classes* in ML. Let's subset the data to only include these, removing all other models. 

We will use the `isin()` function that we also played with in [Class 1 (Section 2.3) ](23-parts-of-speech-analysis):

In [12]:
df = raw_df[raw_df["model"].isin(["human", "chatgpt"])]

### Looking at the Classes

How many of each class do we have in our dataset? We can check this with the `group.by` function, applying `size()` to it:

In [13]:
df.groupby("model").size()

model
chatgpt    26742
human      13371
dtype: int64

:::{admonition} QUESTION
:class: red

Hmm, we seem to have double the amount of `chatgpt` generations as human `generations`. Do you know why that might be a problem?

Consider this with your group and click to reveal answer before proceeding.

```{dropdown} Click to see ANSWER
Classification models generally assume that all classes in a dataset have roughly the same number of examples. Unbalanced classes can cause the classifier to perform poorly on the under-represented class, also called the `minority class` (see {cite:t}`taskiran_comprehensive_2025`).
```
:::

As seen on {numref}`raid-overview`, we have more `chatgpt` rows because different generation parameters are used, producing two sets: greedy and sampling. We'll learn how to deal with unbalanced classes for a bit.

```{admonition} LLM FRAMING: What is "greedy" and "sampling" ? 
:class: dropdown, fuchsia
In brief, `greedy` and `sampling` are *decoding* methods that determine how an LLM selects words when generating text. You will learn more about them later in the course!
```

### Creating a Numerical Label Variable
We want our classifier to predict "human" or "chatgpt", but it doesn't really understand these labels. We'll add the label `is_human` and assign `1` if the row `model` is `human` and `0` if it anything else (such as `chatgpt`):

In [15]:
df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_19654/3636558662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


## 1.3 Create Training Splits!
When training a ML model, we need to consider three splits of the data:
```{figure} ../figures/class2/train_test.png
---
name: train-test-class2
---
Figure from [Rahul Chavan](https://medium.com/@rahulchavan4894/understanding-train-test-and-validation-dataset-split-in-simple-quick-terms-5a8630fe58c8)
```

Common percentage splits for train, val, and test are 60%, 20%, 20% or 70%, 15%, 15%.

### Install scikit-learn
`scikit-learn` is THE Python library for machine learning. It is widely used for everything from supervised learning (classification, regression) to unsupervised learning (clustering), model evaluation and even some pre-processing. The latter is what we'll do now!

Firstly, let's install `scikit-learn`: 

In [17]:
%pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now let's import the `train_test_split`:

In [18]:
from sklearn.model_selection import train_test_split

### Your Turn: Split Data with scikit-learn
Since `raid` has a seperate `test` set, we only need to split our data into `train` and `val`:

:::{admonition} HANDS-ON
:class: red
Instead of showing you the code this time, I'll ask you to check [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) and [this guide](https://medium.com/@whyamit404/understanding-train-test-split-in-pandas-eb1116576c66) to do the following:

1. Use the function `train_test_split()` to split your `df` into `train_df` and `val_df` (remember we have test data in a seperate file!). The size of our validation set should be `20 %`

```{dropdown} Why check documentation?
This is how real coders do their work! While ChatGPT might help you with a code snippet, if you want to understand a function OR use it for a more specific case than the general example, the documentation is a great place to be!
```
:::

:::{admonition} Can you help me breakdown the function?
:class: tip, dropdown

Let's look at the docs together:
```{image} ../figures/class2/train_test_split.png
:alt: train_test_split_docs
:width: 500px
```
&nbsp;
- `arrays`refer to the data variables that should be split. These can be:
    - A complete `df` (our case)
    - Pre-defined numpy arrays `X` and `y` (where `X` refers to the features to be trained on, and `y` the label!)
    - `pandas Series` e.g., `df_balanced["generations"]` for `X` and `df_balanced["is_human"` for `y`.
    - X can also be a df with multiple features e.g., `df["average_sentence_length", "mean_dependency_distance"]`. 
- `test_size` and `train_size` refer to the size of the splits. 
    - We only need to set one of them (it will then compute it self to amount to 100% of the data)
-  The `random_state` ensures that the randomness can be reproduced (ì.e., `seed`), yielding the same split everytime! 
- `shuffle`: The function defaults to shuffling and in many cases we want this! You therefore don't need to specify it.
- `stratify`: Often you would put the class variable to ensure that the shuffle puts an equal amount of each class in each split!

If we insert a `df` directly, we'll get a list of two dataframes that we can unpack:
```python
train_df, val_df = train_test_split(df, ...)
```

Alternatively, if you insert `X` (features) as the first argument and `y` (labels) as the second, the `train_test_split()` **returns** four variables that you unpack as such:
```python
X_train, X_val, y_train, y_val = train_test_split(df["generations"], df["is_human"], ...)
```
:::

Solution if you are stuck or want to compare:

In [19]:
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )